In [ ]:
import pathlib

from matplotlib import pyplot as plt

import el_hd_plasticity.connectomics as cn
from el_hd_plasticity.connectomics import connectivity, datasets, plotting

plt.rcParams['pdf.fonttype'] = 42

%load_ext autoreload
%autoreload 2
%matplotlib inline

figfolder = pathlib.Path('/media/mplitt/SSD_storage/fig_scratch/EL_connectomics')
figfolder.mkdir(parents=True, exist_ok=True)

## Datasets

The male CNS (MCNS) is read from neuPrint; FAFB and BANC (flywire) from Codex (the API token is read from the `CODEX_API_TOKEN` environment
variable; the tables are downloaded once and cached). Codex's `connections_princeton` only contains connections of at least 5 synapses.

In [ ]:
other_datasets = {'MCNS': datasets.NeuprintDataset('MCNS', 'male-cns:v0.9')}
if cn.params.CODEX_TOKEN:
    codex = datasets.CodexClient()
    for name, dataset in (('FAFB', 'fafb'), ('BANC', 'banc')):
        other_datasets[name] = datasets.load_codex(name, dataset, client=codex)
else:
    print('CODEX_API_TOKEN is not set: FAFB and BANC are skipped')

## Connectivity between EL, EPG and ER2/4d

In [ ]:
type_tables, graphs = {}, {}
for name, dataset in other_datasets.items():
    type_tables[name], n_neurons = connectivity.dataset_type_connectivity(dataset)
    graphs[name] = connectivity.connectivity_graph(type_tables[name], n_neurons)
    print(name)
    display(type_tables[name])

In [ ]:
colors = {'EL': plt.cm.Greens(.8), 'EPG': plt.cm.Greys(.8), 'ER2/4d': plt.cm.cool(.7)}
pos = {'ER2/4d': [.2, .4], 'EL': [.7, .7], 'EPG': [.4, .15]}

for name, graph in graphs.items():
    fig, ax = plotting.plot_connectivity_graph(graph, colors, pos, edge_attr='weight')
    ax.set_title(name)
    # fig.savefig(figfolder / f'connectivity_graph_{name}.pdf')